# 1.自检
自检全过。**测试集真值范围 [-2.32, 2.08]** 也说明标准化后分布正常，没有极端离群点。

现在进核心建模。这个脚本同时训练两个模型——**这是整个项目的核心对比**：

| 模型 | 能看到什么 | 对应假设 |
|---|---|---|
| **独立 GP（基线）** | 只有 259 个 Cd 观测 | "各输出互不相关" |
| **LMC（MOGP）** | 259 Cd + 359 Ni + 359 Zn | "输出间相关，可借力" |

两者用完全相同的划分、相同的测试集。RMSE 的差值就是"借力"的价值。## 五个代码里的关键判断，报告要用到

**1. LMC 里我故意没加 `ScaleKernel`——这是个不可辨识性陷阱**

`K = ScaleKernel(RBF) × B`，其中 `ScaleKernel` 有一个 outputscale `σ²`，而 `B` 的对角元也是尺度参数。**两者相乘只能确定一个乘积，单独的值无法确定**——优化器会漫无目的地在 `σ²·B = 常数` 的曲面上游走，收敛变慢且参数无意义。

**独立 GP 基线加了 `ScaleKernel`**，因为它没有 B 来提供尺度。这个不对称是有意的，不是笔误。

**2. `covar_x.mul(covar_i)` 就是 Kronecker 积在长表下的写法**

宽表下 `K = B ⊗ K_input` 是块结构。长表下每行带自己的任务编号，就变成**逐元素相乘**：第 m 行第 n 列 = `K_input(x_m, x_n) × B[i_m, i_n]`。

数学上完全等价，但**长表版本失去了 Kronecker 的计算优势**——你 slides 里讲的 `O((Dn)³) → O(D³+n³)` 那个加速用不上了，因为特征分解需要完整的块结构。这里 N=977，直接 Cholesky 就行。

**这个取舍必须写进报告**：允许缺失（heterotopic）和 Kronecker 加速，**二者不可兼得**。这恰恰解释了为什么真实场景需要 SVGP。

**3. `RANK=2` 是我的初始选择，你后面要做敏感性分析**

`rank` 就是 LMC 的 Q。P=3 时 rank 最多取 3。**跑完先看 rank=1/2/3 分别的 RMSE**——这是报告里现成的一张表，而且能讨论"多少个潜在函数够用"。

**4. 原始 ppm 尺度算了两个 RMSE，数字会不一样**

`exp(后验均值)` 是**中位数**的还原，`exp(μ+σ²/2)` 是**期望**的还原。文献里两种都有人用，报告里说清楚你用哪个即可。**不说清楚会被质疑数字对不上文献。**

**5. 最关键的看点：学到的 B 相关矩阵 vs EDA 的经验相关性**

EDA 里 Cd-Ni=0.65、Cd-Zn=0.69、Ni-Zn=0.70。如果模型学出来的 B 相关矩阵接近这几个数，**说明模型确实捕捉到了真实的跨输出结构**——这是你报告里最有说服力的一张图。

如果差很远（比如全接近 1 或全接近 0），说明优化没收敛好，需要调 `LR` 或 `N_ITER`。

---

跑 `python src/03_train_lmc.py`，把这几样发我：**两个模型的 RMSE、改善百分比、学到的 B 相关矩阵、lengthscale**。这就是你报告里第一组真正的结果了。

# 2. 结果
## 先看好消息：B 矩阵学得很准

| 配对 | EDA 经验相关性 | 模型学到的 B | 差距 |
|---|---|---|---|
| Cd–Ni | 0.65 | 0.753 | +0.10 |
| Cd–Zn | 0.69 | 0.718 | +0.03 |
| Ni–Zn | 0.70 | 0.762 | +0.06 |

**模型在完全不知道经验相关性的情况下，自己从数据里学出了几乎相同的跨输出结构。** 这张对比表直接放进报告，是"模型确实捕捉到了真实结构"的最强证据。

（系统性略高是正常的：B 刻画的是**去掉噪声后的潜在函数相关性**，而 EDA 的 Pearson 相关是**含噪观测**的相关性，噪声会稀释相关系数。这一句解释也可以写进报告。）

---

## 但有个严重问题：**lengthscale 崩塌了**

```
学到的空间 lengthscale: X=0.074 km, Y=0.058 km
EDA 测得的平均采样间距: ≈ 0.27 km
```

**lengthscale 只有采样间距的 1/4。** 这意味着 RBF kernel 在任意两个**不同**采样点之间的相关性 ≈ `exp(-(0.27/0.07)²/2) ≈ 0.0005` —— 基本为零。

### 这个数字的含义，比它看起来严重

**你的模型现在不是空间高斯过程，而是一个逐点回归。**

它是这样工作的：测试位置 x 上，Cd 未观测，但 Ni 和 Zn 在**完全相同的坐标**上有观测（距离 = 0，`K_input = 1`）。于是模型完全靠 `B[Cd,Ni]=0.66` 和 `B[Cd,Zn]=0.61` 这两个数，从同点的 Ni/Zn 值推 Cd。**周围邻居的信息一点没用上。**

**旁证**：独立 GP 的 RMSE = 0.9715。标准化数据上，**直接预测均值 0 的 RMSE 就是 1.0**。也就是说独立 GP 几乎什么都没学到——它的 lengthscale 大概率也崩塌了，退化成了常数预测器（噪声方差 0.2862 相对偏高也支持这个判断）。

**如果报告里不处理这件事，会被问穿**：你的 23% 改善，来源不是"多输出 GP 的空间协同建模"，而是"同点线性回归"——那用普通线性回归就够了，何必上 GP。

---

## 现在要做的三件事（按顺序）

**1. 确认独立 GP 的 lengthscale**

在 `03` 脚本里补一行打印：
```python
ls_gp = gp.covar_module.base_kernel.lengthscale.detach().numpy().ravel()
print(f"独立GP lengthscale (km): X={ls_gp[0]:.3f}, Y={ls_gp[1]:.3f}")
```
如果也是 0.0x 量级 → 确认两个模型都崩塌了。

**2. 检验这是不是局部最优（关键实验）**

把 `LS_INIT` 分别设成 `0.5 / 1.0 / 2.0` 各跑一次，记录**最终 loss** 和**测试 RMSE**：

- 如果大初值收敛到同样的小 lengthscale → 数据本身就没有可利用的空间结构，这是**真实发现**，报告里如实写
- 如果大初值停在较大 lengthscale 且 **loss 更低** → 之前是局部最优，用新结果

**3. 加一个 lengthscale 先验，防止崩塌**

```python
lmc.covar_module.register_prior(
    "lengthscale_prior",
    gpytorch.priors.GammaPrior(3.0, 6.0),   # 众数约 0.33 km, 尾部允许到 1km+
    "lengthscale",
)
```
这在地统计里是标准做法——**先验编码了"空间相关距离应该在采样间距量级"这个领域知识**，也是报告里能写的一个方法学要点。

---

## 这件事对报告的正面价值

**别把它当成 bug，它是你报告里最有深度的一段讨论。**

Jura 的 Cd 空间自相关本来就弱（地统计文献里 Cd 的变差函数块金效应显著），而跨变量相关性强（0.65-0.70）。**所以在这个数据集上，"跨输出借力"确实压倒了"空间平滑"——这本身就是一个有内容的结论**，而不是模型没调好。

但你必须**主动做实验证明**这一点（第 2 步），而不是让读者自己发现 lengthscale 很小然后怀疑你。

---

跑完 `LS_INIT` 的三组对比发我，我们据此决定是修模型还是把这个现象写成发现。